In [ ]:
import pandas as pd
from difflib import get_close_matches
from tqdm.notebook import tqdm

In [ ]:
data = pd.read_csv("t1_ours_annotated.tsv", sep='\t', encoding="cp1252")
data.head()

In [ ]:
data[data.examples.str.contains("Berlin")].examples.iloc[0]

In [ ]:
data.shape

In [ ]:
tqdm.pandas()

In [ ]:
data[~data.full_phrase.isna()]

In [ ]:
data.iloc[910].examples

In [ ]:
from numpy import nan
import re

In [ ]:
example_pat = re.compile(r"\([\"'](.+?)[\"'], '(\d+)', (\d{4})\)")

In [ ]:
data["isinstr"] = data.apply(lambda x: x.full_phrase in x.examples if x.full_phrase is not nan else True, axis = 1)

In [ ]:
data[~data.isinstr]

In [ ]:
data.iloc[4].full_phrase

In [ ]:
data.iloc[4].examples

In [ ]:
data.iloc[5].examples

In [ ]:
parsed = pd.read_parquet("sentences_parsed.parquet")
print(parsed.head())

In [ ]:
sent = "Luxembourg, 17.03.2017 Environmental and Social Data Sheet Overview Project Name: VASAKRONAN NEARLY ZERO ENERGY BUILDINGS Project Number: 2016-0842 Country: Sweden Project Description: The proposed project consists of the development of two large mixed-use buildings supporting the Swedish strategy for the development of nearly zero-energy buildings (NZEB)."

In [ ]:
parsed[parsed.doc_id == "139351344"].sentence

In [ ]:
assert parsed.iloc[46242].sentence == sent

In [ ]:
row = parsed.iloc[46242]
for i, (lemma, upos, dep, head) in enumerate(
    zip(row.lemmas.split(), row.uposes.split(), row.deps.split(), row.heads.split())
):
    print(f"{i+1} {lemma} {upos} {dep} {head}")

In [ ]:
from flair.data import Sentence
from flair.models import SequenceTagger

# load tagger
tagger = SequenceTagger.load("flair/chunk-english")

# make example sentence
sentence = Sentence("The happy man has been eating at the diner")

# predict NER tags
tagger.predict(sentence)

# print sentence
print(sentence)

# print predicted NER spans
print('The following NER tags are found:')
# iterate over entities and print
for entity in sentence.get_spans('np'):
    print(entity)


In [ ]:
# make example sentence
sentence = Sentence(sent)

# predict NER tags
tagger.predict(sentence)

# print sentence
print(sentence)

# print predicted NER spans
print('The following NER tags are found:')
# iterate over entities and print
for entity in sentence.get_spans():
    print(dir(entity))

In [ ]:
for entity in sentence.get_spans():
    print(entity)

In [ ]:
def find_long_nps(sent):
    sentence = Sentence(sent)
    tagger.predict(sentence)
    last_tag = ""
    nps = set()
    current_np = []
    for entity in sentence.get_spans():
        if entity.tag == "NP":
            if last_tag!="PP":
                nps.add(' '.join([ent.text for ent in current_np]))
                current_np = [entity]
            else:
                current_np.append(entity)
            last_tag = "NP"
        elif (entity.tag == "PP") and (last_tag == "NP"):
            current_np.append(entity)
            last_tag = "PP"
        else:
            nps.add(' '.join([ent.text for ent in current_np]))
            current_np = []
            last_tag = entity.tag
    return nps

In [ ]:
def clean_example(example):
    try:
        return [(match.group(1), match.group(2), match.group(3)) for match in re.finditer(example_pat, example)]
    except AttributeError:
        print(example)
        return example

In [ ]:
get_close_matches('Swedish strategy for the development of nearly zero-energy buildings (NZEBs)', nps)

In [ ]:
long_phrases = data[~data.full_phrase.isna()]

In [ ]:
import json
import gzip
with gzip.open("green-long.json.gz", "rt") as f:
    for line in f:
        line = json.loads(line
                         )
        print(json.dumps(line, ensure_ascii=False))

In [ ]:
with open("green-long.json", "w") as f:
    for i, row in long_phrases.iterrows():
        for (ex, doc_id, year) in clean_example(row.examples):
            if not row.full_phrase in ex:
                nps = find_long_nps(ex)

                for target in nps:
                    if target:
                        f.write(
                            json.dumps(
                            {"target": target.rstrip("."), "example": ex, "year": year, "doc_id": doc_id,
                             "full_phrase": row.full_phrase}, ensure_ascii=False,
                        ) + "\n"
                               )
            else:
                f.write(
                            json.dumps(
                            {"target": row.full_phrase, "example": ex, "year": year, "doc_id": doc_id,
                             "full_phrase": row.full_phrase}, ensure_ascii=False,
                        ) + "\n"
                               )
        if i % 10 == 0:
            print(i)

In [ ]:
{"target": "different sector", "example": "This is a multi-scheme programme comprising different sectors geographically spread over Cyprus.", "year": 2014, "doc_id": "47726845"}
{"target": "additional CO2", "example": "The new facilities will create additional CO2 emissions that will be compensated by the reduction of emissions of the refurbished buildings.",
 "year": 2014, "doc_id": "47826590"}

In [ ]:
long_phrases['target']

In [ ]:
long_phrases["lengths"] = long_phrases.target.apply(lambda x: len(x))

In [ ]:
for i, row in long_phrases[long_phrases.lengths>1].iterrows():
    print(row.target)
    print(row.full_phrase)
    print("----------")

In [ ]:
for i, row in long_phrases[long_phrases.lengths==0].iterrows():
    print(row.target)
    print(row.full_phrase)
    print(row.examples)
    print("----------")

In [ ]:
long_phrases[long_phrases.lengths==0]